# YOLO 3회차 산업 데모 ③ — 분류(classify)로 주조 부품 불량 판정

제조 라인 끝에서 카메라가 부품을 **한 개씩** 찍는다면, 박스를 그릴 필요 없이 "이 사진은 정상인가 불량인가"만 답하면 됩니다. 이것이 **이미지 분류(classification — 사진 한 장에 클래스 하나를 매기는 작업)** 입니다.
탐지보다 라벨링이 훨씬 쉽고(폴더에 나눠 담기만 하면 됨), 모델도 빠릅니다.

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| Step 1 | 탐지와 분류의 차이 — 사전학습 분류 모델 써 보기 | top-5 예측 |
| Step 2 | 주조 부품 데이터 받기 (캐글) | 클래스별 장 수, 샘플 그림 |
| Step 3 | 불량 판정 모델 미세조정 | `cast_cls_best.pt` |
| Step 4 | 평가 — 정확도와 혼동 행렬 | `cls_confusion.png` |
| Step 5 | 규칙 — 불량 확률 3구간(통과 / 재검사 / 불량) | `cls_threshold.csv`, `cls_errors.png` |

**데이터**: 캐글(Kaggle, 데이터셋 공유 사이트) `ravirajsinh45/real-life-industrial-dataset-of-casting-product` — 펌프 부품(임펠러) 주조품을 위에서 찍은 300×300 흑백 사진, 정상 `ok_front` / 불량 `def_front` 두 폴더. 라이선스는 데이터셋 페이지에서 확인하고 학습·실습 용도로만 씁니다.
**실행 환경**: Google Colab, 런타임 유형 **T4 GPU**(Colab에서 빌려 쓰는 NVIDIA 그래픽카드)

## 0. 준비

In [ ]:
# ultralytics(YOLO), kagglehub(캐글 데이터 다운로드). -q: 설치 로그 줄임
!pip -q install ultralytics kagglehub

In [ ]:
import os, json, shutil, base64, subprocess, time
from pathlib import Path

# 영상·이미지 처리(cv2), 배열(numpy), 표(pandas), 그래프(matplotlib)
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display  # 셀 출력에 HTML(영상 플레이어 등) 띄우기
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())  # T4 런타임이면 True

try:
    from google.colab import drive  # Colab에만 있는 모듈 → 성공하면 Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True  # False면 Drive 대신 런타임 임시 폴더에 저장 (연결이 끊기면 사라짐)
if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day3/cls')  # Drive 안 결과 폴더
else:
    SAVE_DIR = Path('outputs/cls')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 중간 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# OpenCV(BGR) 이미지를 matplotlib로 보여 주기
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 축을 뒤집어 BGR→RGB, 눈금 숨김
    if title:
        plt.title(title)
    plt.show()

# 여러 장을 격자로 보여 주기 (입력: BGR 이미지 목록, 제목 목록, 한 줄에 놓을 장 수)
def show_grid(imgs, titles=None, cols=3, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 줄 수 (올림)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for k, im in enumerate(imgs):
        ax = axes.ravel()[k]
        ax.imshow(im[..., ::-1])
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

# mp4 영상을 노트북 안에서 재생 (입력: 영상 경로, 표시 폭 px)
# OpenCV가 만든 mp4v 영상은 브라우저가 못 틀어서 H.264로 다시 인코딩
def show_video(path, width=640):
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상을 글자(base64)로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# 파일을 SAVE_DIR로 복사하고 복사된 경로를 돌려줌
def save_out(path):
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

---
## Step 1. 탐지와 분류의 차이

| | 탐지(detect) | 분류(classify) |
|---|---|---|
| 답 | 물체마다 박스 + 클래스 | 사진 전체에 클래스 하나 |
| 라벨 | 박스 좌표를 그려야 함 | 클래스 이름 폴더에 사진을 넣기만 하면 됨 |
| 모델 이름 | `yolo26n.pt` | `yolo26n-cls.pt` (`-cls`가 붙음) |
| 결과 위치 | `r.boxes` | `r.probs` (클래스별 확률) |

먼저 ImageNet(1,000종 사물 사진으로 만든 분류 데이터셋)으로 사전학습된 분류 모델에 사진 한 장을 넣어 봅니다.

In [ ]:
cls_model = YOLO('yolo26n-cls.pt')  # 처음 실행 시 자동 다운로드
from ultralytics.utils import ASSETS  # ultralytics 패키지 안에 들어 있는 예시 사진 폴더
r = cls_model(str(ASSETS / 'bus.jpg'), verbose=False)[0]  # 예시 사진 (버스)
# probs.top5: 확률이 높은 클래스 번호 5개, probs.top5conf: 그 확률
for i, p in zip(r.probs.top5, r.probs.top5conf.cpu().numpy()):
    print(f'{cls_model.names[i]:<20} {p:.3f}')
show_bgr(r.orig_img, f'top1: {cls_model.names[r.probs.top1]}', size=(5, 6))

사진 전체에 대해 확률을 매길 뿐, **어디에** 있는지는 알려 주지 않습니다. 부품이 사진 가운데 한 개씩 놓이는 검사 라인에서는 이것으로 충분합니다.

---
## Step 2. 주조 부품 데이터 받기

캐글 데이터는 `kagglehub`로 받습니다. 공개 데이터셋은 로그인 없이 받아지는 경우가 많고, 오류가 나면 Colab 왼쪽 **보안 비밀(열쇠 아이콘)** 에 `KAGGLE_API_TOKEN`을 등록한 뒤 다시 실행하세요(워크숍 안내서의 방법과 같습니다).

Ultralytics 분류 학습은 다음 **폴더 구조**만 맞으면 됩니다. 라벨 파일이 따로 없습니다.
```
cast_data/
  train/ defect/ *.jpeg   ok/ *.jpeg
  val/   defect/ *.jpeg   ok/ *.jpeg
```

In [ ]:
import kagglehub
KAGGLE_SLUG = 'ravirajsinh45/real-life-industrial-dataset-of-casting-product'

try:
    from google.colab import userdata  # Colab 보안 비밀 읽기
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    pass  # 보안 비밀이 없으면 로그인 없이 시도

SRC = Path(kagglehub.dataset_download(KAGGLE_SLUG))  # 내려받은 폴더 경로
print('받은 위치:', SRC)
# 'train' 폴더 안에 def_front·ok_front가 있는 위치를 찾음 (압축 구조가 바뀌어도 동작하도록)
TRAIN_SRC = next(p for p in SRC.rglob('train') if (p / 'def_front').is_dir())
TEST_SRC = TRAIN_SRC.parent / 'test'
print('train:', TRAIN_SRC, '| test:', TEST_SRC)

In [ ]:
# Ultralytics 분류 폴더 구조로 정리 (클래스 이름을 영문 짧은 이름으로: def_front → defect, ok_front → ok)
# 복사 대신 심볼릭 링크(symlink — 원본을 가리키는 바로가기 파일)를 만들어 디스크와 시간을 아낌
DATA = Path('cast_data')
RENAME = {'def_front': 'defect', 'ok_front': 'ok'}
for split, src in [('train', TRAIN_SRC), ('val', TEST_SRC)]:  # 원본 test를 검증(val)으로 사용
    for old, new in RENAME.items():
        dst = DATA / split / new
        dst.mkdir(parents=True, exist_ok=True)
        for f in (src / old).glob('*.jpeg'):
            link = dst / f.name
            if not link.exists():
                link.symlink_to(f.resolve())

counts = pd.DataFrame([{'split': s, 'class': c, 'images': len(list((DATA / s / c).glob('*')))}
                       for s in ['train', 'val'] for c in RENAME.values()])
display(counts.pivot(index='class', columns='split', values='images'))

# 클래스별 샘플 4장씩
samples = []
for c in ['ok', 'defect']:
    for f in sorted((DATA / 'train' / c).glob('*'))[:4]:
        samples.append((cv2.imread(str(f)), c))
show_grid([s[0] for s in samples], [s[1] for s in samples], cols=4, size=3)

**확인해 보기** — 불량 사진에서 어떤 결함(구멍, 깨진 테두리, 표면 거칠음)이 보이나요? 결함이 작고 다양할수록 분류가 어려워지고, 이때는 결함 위치를 그리는 탐지·분할 모델이 필요해질 수 있습니다.

---
## Step 3. 불량 판정 모델 미세조정

사전학습 분류 모델(`yolo26n-cls.pt`)에서 시작해 정상/불량 두 클래스로 다시 학습합니다(미세조정 — 사전학습 모델을 내 데이터로 조금 더 학습시키는 것).
분류 모델은 보통 **224×224**로 줄여 넣습니다. 이 데이터는 T4에서 epoch당 30초 안팎입니다.

In [ ]:
QUICK_RUN = True  # ✏️ True: 3 epoch 빠른 확인 / False: 15 epoch
EPOCHS = 3 if QUICK_RUN else 15  # epoch(에폭): 학습 데이터 전체를 한 번 다 보는 단위
t0 = time.time()
m = YOLO('yolo26n-cls.pt')
# data=폴더 경로: 분류는 data.yaml 없이 폴더를 그대로 넘김
m.train(data=str(DATA.resolve()), epochs=EPOCHS, imgsz=224, batch=64, project='runs_cls', name='cast',
        exist_ok=True, seed=0, verbose=False)
BEST = Path(m.trainer.save_dir) / 'weights' / 'best.pt'  # 검증 정확도가 가장 좋았던 가중치
shutil.copy(BEST, SAVE_DIR / 'cast_cls_best.pt')
print(f'학습 {(time.time() - t0) / 60:.1f}분 | {BEST}')

---
## Step 4. 평가 — 정확도와 혼동 행렬

검증 사진 전부를 예측해 **정확도(accuracy — 맞힌 사진 수 ÷ 전체)** 와 **혼동 행렬(정답 클래스 × 예측 클래스 표)** 을 직접 계산합니다.
품질 검사에서 중요한 칸은 두 개입니다.
- **놓친 불량**: 정답 불량 → 예측 정상. 불량품이 고객에게 갑니다(가장 비쌈).
- **과검**: 정답 정상 → 예측 불량. 멀쩡한 부품을 버리거나 재검사합니다.

In [ ]:
best = YOLO(BEST)
CLASS_ID = {v: k for k, v in best.names.items()}  # 클래스 이름 → 번호
DEF = CLASS_ID['defect']

# 검증 사진을 예측해 '불량일 확률'을 한 줄씩 기록
rows = []
for c in ['ok', 'defect']:
    files = sorted((DATA / 'val' / c).glob('*'))
    for k in range(0, len(files), 64):  # 64장씩 묶어 예측 (빠름)
        for f, r in zip(files[k:k + 64], best([str(f) for f in files[k:k + 64]], verbose=False)):
            rows.append({'file': f.name, 'true': c, 'p_defect': round(float(r.probs.data[DEF]), 4)})
val = pd.DataFrame(rows)
val['pred'] = np.where(val['p_defect'] >= 0.5, 'defect', 'ok')  # 기본 기준: 확률 0.5
acc = (val['true'] == val['pred']).mean()
cm = pd.crosstab(val['true'], val['pred'], rownames=['true'], colnames=['pred'])  # 혼동 행렬
print(f'검증 {len(val)}장 | 정확도 {acc:.3f}')
display(cm)

# 혼동 행렬 그림
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(cm.values, cmap='Blues')
for (i, j), v in np.ndenumerate(cm.values):
    ax.text(j, i, v, ha='center', va='center', fontsize=12)
ax.set_xticks(range(cm.shape[1])); ax.set_xticklabels(cm.columns); ax.set_xlabel('pred')
ax.set_yticks(range(cm.shape[0])); ax.set_yticklabels(cm.index); ax.set_ylabel('true')
ax.set_title(f'accuracy {acc:.3f}')
plt.tight_layout(); plt.savefig('cls_confusion.png', dpi=120); save_out('cls_confusion.png'); plt.show()

---
## Step 5. 규칙 — 불량 확률 3구간 판정

모델의 답을 "0.5 넘으면 불량" 하나로 쓰지 않고, 현장 규칙으로 바꿉니다.

| 구간 | 조건 | 처리 |
|---|---|---|
| 통과 | `p_defect < LOW` | 자동 합격 |
| 재검사 | `LOW ≤ p_defect < HIGH` | 사람이 다시 봄 |
| 불량 | `p_defect ≥ HIGH` | 자동 불합격 |

`LOW`를 낮출수록 놓친 불량이 줄고, 대신 재검사 물량이 늘어납니다. 아래 표로 두 값의 균형을 봅니다.

In [ ]:
# 기준값 조합별로 '놓친 불량', '과검', '재검사 비율'을 계산
def three_zone(low, high):
    zone = np.select([val['p_defect'] < low, val['p_defect'] >= high], ['pass', 'reject'], 'recheck')
    is_def = val['true'] == 'defect'
    return {'low': low, 'high': high,
            'missed_defect': int(((zone == 'pass') & is_def).sum()),     # 불량인데 자동 합격
            'false_reject': int(((zone == 'reject') & ~is_def).sum()),   # 정상인데 자동 불합격
            'recheck_ratio': round(float((zone == 'recheck').mean()), 3)}  # 사람이 볼 비율

thr = pd.DataFrame([three_zone(lo, hi) for lo in [0.05, 0.1, 0.2, 0.3, 0.5] for hi in [0.5, 0.7, 0.9] if lo <= hi])
thr.to_csv('cls_threshold.csv', index=False); save_out('cls_threshold.csv')  # 열 이름은 영문
thr

In [ ]:
LOW, HIGH = 0.1, 0.9  # ✏️ 위 표를 보고 고르기 — 놓친 불량 0을 지키면서 재검사 비율이 작은 조합
pick = three_zone(LOW, HIGH)
print(f'LOW {LOW}, HIGH {HIGH} → 놓친 불량 {pick["missed_defect"]}장, 과검 {pick["false_reject"]}장, 재검사 {pick["recheck_ratio"]:.1%}')

# 틀리거나 애매한 사진 보기: 놓친 불량 → 재검사 구간 순으로 최대 8장
val['zone'] = np.select([val['p_defect'] < LOW, val['p_defect'] >= HIGH], ['pass', 'reject'], 'recheck')
hard = pd.concat([val[(val.zone == 'pass') & (val.true == 'defect')],
                  val[(val.zone == 'reject') & (val.true == 'ok')],
                  val[val.zone == 'recheck'].sort_values('p_defect')]).head(8)
imgs = [cv2.imread(str(DATA / 'val' / t / f)) for f, t in zip(hard.file, hard.true)]
show_grid(imgs, [f'true {t} | p_defect {p:.2f} | {z}' for t, p, z in zip(hard.true, hard.p_defect, hard.zone)], cols=4, size=3)
if imgs:
    canvas = np.hstack([cv2.resize(im, (200, 200)) for im in imgs])  # 가로로 이어 붙여 한 장으로 저장
    cv2.imwrite('cls_errors.png', canvas); save_out('cls_errors.png')

**생각해 보기** — 재검사 비율 10%가 현장에서 감당할 만한가요? 하루 1만 개 생산이면 1,000개를 사람이 봐야 합니다. 이 숫자가 바로 "모델을 더 좋게 만들 가치"를 계산하는 기준이 됩니다.

---
## 도전 과제

1. **에폭 늘리기** — `QUICK_RUN = False`로 다시 학습해 재검사 비율이 얼마나 줄어드는지 비교해 보세요.
2. **설명 그림** — 재검사 구간 사진을 모아 "모델이 헷갈리는 결함 유형"을 3가지로 분류해 보세요. 데이터를 더 모을 방향이 나옵니다.
3. **워크숍 주제와 연결** — 재활용 쓰레기(T3)나 작물 해충(T7)처럼 사진 한 장에 대상이 하나인 경우라면, 탐지 대신 분류로 시작하는 것이 더 빠를 수 있습니다. 폴더만 나누면 바로 학습됩니다.

## 정리

- 분류는 사진 **한 장에 답 하나**. 라벨은 클래스 이름 폴더에 사진을 넣는 것으로 끝납니다.
- 결과는 `r.probs`(클래스별 확률)에서 꺼냅니다.
- 품질 검사는 **놓친 불량**과 **과검**의 비용이 다릅니다. 확률을 한 기준으로 자르지 말고 3구간 규칙으로 사람 검사와 나눕니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| 분류 (classification) | 사진 한 장 전체에 클래스 하나를 매기는 작업입니다. 위치(박스)는 내지 않습니다. |
| `-cls` 모델 | 분류용 YOLO 모델입니다. `yolo26n-cls.pt`는 ImageNet으로 사전학습되어 있습니다. |
| ImageNet | 1,000종 사물 사진으로 만든 대표적인 분류 데이터셋입니다. |
| probs / top1 / top5 | 클래스별 확률, 가장 높은 클래스, 상위 5개 클래스입니다. |
| 미세조정 | 사전학습 모델을 내 데이터로 조금 더 학습시키는 것입니다. |
| 심볼릭 링크 | 원본 파일을 가리키는 바로가기 파일입니다. 복사하지 않아 빠르고 공간을 덜 씁니다. |
| imgsz | 학습·예측 때 이미지를 맞추는 한 변 크기(픽셀)입니다. 분류는 보통 224입니다. |
| epoch | 학습 데이터 전체를 한 번 모두 학습하는 단위입니다. |
| 정확도 (accuracy) | 맞힌 사진 수 ÷ 전체 사진 수입니다. 클래스 수가 크게 다르면 이것만으로는 부족합니다. |
| 혼동 행렬 | 정답 클래스(행)와 예측 클래스(열)를 교차해 장 수를 센 표입니다. 어떤 실수를 하는지 보입니다. |
| 놓친 불량 / 과검 | 불량을 정상으로 판정한 것 / 정상을 불량으로 판정한 것입니다. 현장 비용이 서로 다릅니다. |
| 3구간 판정 | 확률 두 기준(LOW, HIGH)으로 통과·재검사·불량을 나누는 규칙입니다. |
| kagglehub | 캐글 데이터셋을 코드 한 줄로 내려받는 라이브러리입니다. |